# Sprint 2 — Milestone 6: Research Contribution & Advanced Analytics

- SCT213-C002-0141/2024 GACHOMO NJERU
- SCT213-C002-0063/2024 MWITA NYEHITA
- SCT213-C002-0142/2024 KARWEGA NJERU

**Objectives:**
- Introduce a novel machine-learning contribution: tip-generosity prediction and customer segmentation
- Apply SHAP (SHapley Additive exPlanations) for model explainability — a research-level technique
- Evaluate Random Forest, Gradient Boosting, and Logistic Regression models comparatively
- Perform K-Means clustering to discover latent customer archetypes
- Demonstrate a fully integrated pipeline from raw data → insight → decision support
- Produce a research-level evaluation suitable for a conference-standard paper

## 0. Environment Setup & Data Reconstruction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- Core ML libraries ---
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, GridSearchCV, learning_curve)
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay,
                              mean_squared_error, mean_absolute_error, r2_score)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# --- SHAP for explainability ---
try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP library loaded successfully")
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed — run: pip install shap")

# --- Random seed for full reproducibility ---
SEED = 42
np.random.seed(SEED)

# --- Global plot style (matches Milestone 3) ---
plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'font.family'       : 'DejaVu Sans',
    'axes.titlesize'    : 13,
    'axes.labelsize'    : 11,
})
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
print("Environment ready.")

In [ ]:
# ── Reproduce full Sprint-2 pipeline ──────────────────────────────────────────
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
df_raw = pd.read_csv(url)
df = df_raw.copy()

# Feature engineering (identical to Milestones 2–5)
df['tip_pct']          = (df['tip'] / df['total_bill'] * 100).round(2)
df['bill_per_person']  = (df['total_bill'] / df['size']).round(2)
day_avg                = df.groupby('day')['tip'].mean().round(2).reset_index()
day_avg.columns        = ['day', 'day_avg_tip']
df                     = pd.merge(df, day_avg, on='day', how='left')
df['tip_vs_day_avg']   = (df['tip'] - df['day_avg_tip']).round(2)
df['is_weekend']       = df['day'].isin(['Sat', 'Sun']).astype(int)

def tip_category(pct):
    if pct < 10:   return 'Low'
    elif pct < 20: return 'Average'
    else:          return 'Generous'

df['tip_category']   = df['tip_pct'].apply(tip_category)
df['sex_encoded']    = (df['sex']    == 'Male').astype(int)
df['smoker_encoded'] = (df['smoker'] == 'Yes').astype(int)
df['time_encoded']   = (df['time']   == 'Dinner').astype(int)
df['day_num']        = df['day'].map({'Thur':1, 'Fri':2, 'Sat':3, 'Sun':4})

print(f"Dataset ready | Shape: {df.shape}")
print(f"Tip category distribution:\n{df['tip_category'].value_counts()}")
df.head()

---
## 1. Novel Contribution — Overview

Milestones 1–5 established descriptive and inferential analytics on the restaurant tips dataset.
Milestone 6 introduces **three novel research-level contributions** that extend the system
beyond classical statistics:

| # | Contribution | Technique | Research Value |
|---|---|---|---|
| **C1** | Tip-generosity prediction | Random Forest, Gradient Boosting, Logistic Regression — comparative evaluation | Predictive modelling of categorical outcome |
| **C2** | Model explainability | SHAP (SHapley Additive exPlanations) | Transparent AI — aligns with XAI research agenda |
| **C3** | Customer segmentation | K-Means clustering + PCA visualisation | Unsupervised discovery of latent archetypes |

Together, these move the system from **descriptive** (what happened?) to **predictive** (what will happen?) and **prescriptive** (what should we do?) analytics.

---
## 2. Feature Engineering for Machine Learning

All features are derived from the original dataset — no external data sources required.
This demonstrates how thoughtful feature engineering can extract predictive signal from a small dataset.

In [ ]:
# ── Feature matrix and target vector ──────────────────────────────────────────
FEATURES = [
    'total_bill',       # Raw bill amount — strongest predictor
    'size',             # Party size
    'bill_per_person',  # Normalised spend — controls for party size
    'sex_encoded',      # 1=Male, 0=Female
    'smoker_encoded',   # 1=Smoker, 0=Non-smoker
    'time_encoded',     # 1=Dinner, 0=Lunch
    'is_weekend',       # 1=Sat/Sun, 0=Thur/Fri
    'day_num',          # Ordinal: Thur=1 … Sun=4
]

# Multi-class target (3 classes)
label_map    = {'Low': 0, 'Average': 1, 'Generous': 2}
inv_label    = {v: k for k, v in label_map.items()}
df['target'] = df['tip_category'].map(label_map)

X = df[FEATURES].values
y = df['target'].values

# Standardise (required for Logistic Regression and SVM; harmless for tree-based)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / test split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=SEED, stratify=y)

print(f"Training set : {X_train.shape[0]} rows")
print(f"Test set     : {X_test.shape[0]} rows")
print(f"\nClass distribution (train):")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {inv_label[u]:10s}: {c} ({c/len(y_train)*100:.1f}%)")

---
## 3. Comparative Model Evaluation (C1)

Three classifiers are benchmarked under identical conditions using 5-fold stratified cross-validation.
This mirrors the comparative evaluation methodology used in conference machine learning papers.

In [ ]:
# ── Model definitions ─────────────────────────────────────────────────────────
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=SEED, C=1.0),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=6,
                                                    random_state=SEED, class_weight='balanced'),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=150, learning_rate=0.08,
                                                         max_depth=4, random_state=SEED),
}

# ── 5-Fold Stratified Cross-Validation ───────────────────────────────────────
cv   = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = {}

print("=" * 65)
print(f"{'Model':<25} {'Mean Acc':>10} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 65)

for name, clf in models.items():
    scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<25} {scores.mean():.4f}     {scores.std():.4f}   "
          f"{scores.min():.4f}   {scores.max():.4f}")

print("=" * 65)
best_model_name = max(cv_results, key=lambda k: cv_results[k].mean())
print(f"\nBest model by mean CV accuracy: {best_model_name}")

In [ ]:
# ── Cross-validation results chart ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))

positions = np.arange(len(cv_results))
colors    = ['#2196F3', '#4CAF50', '#FF5722']

for i, (name, scores) in enumerate(cv_results.items()):
    ax.bar(i, scores.mean(), color=colors[i], alpha=0.85, edgecolor='white',
           label=f"{name} ({scores.mean():.3f}±{scores.std():.3f})")
    ax.errorbar(i, scores.mean(), yerr=scores.std(),
                fmt='none', color='#333', capsize=6, capthick=2, lw=2)
    # Plot individual fold dots
    ax.scatter([i]*5, scores, color='white', s=30, zorder=5, edgecolors=colors[i], lw=1.5)

ax.set_xticks(positions)
ax.set_xticklabels(cv_results.keys())
ax.set_ylabel('Accuracy (5-Fold CV)')
ax.set_title('Comparative Model Accuracy — 5-Fold Stratified Cross-Validation', fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig('m6_fig01_cv_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: m6_fig01_cv_comparison.png")

In [ ]:
# ── Held-out test set evaluation ─────────────────────────────────────────────
test_results = {}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred   = clf.predict(X_test)
    y_proba  = clf.predict_proba(X_test) if hasattr(clf, 'predict_proba') else None
    acc      = accuracy_score(y_test, y_pred)
    auc      = roc_auc_score(y_test, y_proba, multi_class='ovr') if y_proba is not None else None
    test_results[name] = {'acc': acc, 'auc': auc, 'y_pred': y_pred, 'y_proba': y_proba, 'clf': clf}

print("=" * 55)
print(f"{'Model':<25} {'Test Acc':>10} {'ROC-AUC':>10}")
print("-" * 55)
for name, r in test_results.items():
    auc_str = f"{r['auc']:.4f}" if r['auc'] else "N/A"
    print(f"{name:<25} {r['acc']:.4f}     {auc_str}")
print("=" * 55)

In [ ]:
# ── Confusion matrices for all three models ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = ['Low', 'Average', 'Generous']
cmaps = ['Blues', 'Greens', 'Oranges']

for ax, (name, r), cmap in zip(axes, test_results.items(), cmaps):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap=cmap, colorbar=False)
    ax.set_title(f"{name}\nAcc={r['acc']:.3f}", fontweight='bold', fontsize=10)

plt.suptitle('Confusion Matrices — Held-Out Test Set (n=61)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('m6_fig02_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: m6_fig02_confusion_matrices.png")

In [ ]:
# ── Detailed classification report for best model ────────────────────────────
best = best_model_name
print(f"Classification Report — {best}")
print("=" * 60)
print(classification_report(
    y_test,
    test_results[best]['y_pred'],
    target_names=class_names
))

In [ ]:
# ── One-vs-Rest ROC curves for all models ────────────────────────────────────
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(y_test, classes=[0, 1, 2])
class_names_long = ['Low (<10%)', 'Average (10-20%)', 'Generous (>20%)']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
line_styles = ['-', '--', ':']

for ax, (model_name, r) in zip(axes, test_results.items()):
    if r['y_proba'] is None:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
        continue
    for cls_idx, (cls_name, ls) in enumerate(zip(class_names_long, line_styles)):
        fpr, tpr, _ = roc_curve(y_bin[:, cls_idx], r['y_proba'][:, cls_idx])
        auc_cls     = roc_auc_score(y_bin[:, cls_idx], r['y_proba'][:, cls_idx])
        ax.plot(fpr, tpr, ls=ls, lw=2, label=f"{cls_name}\n(AUC={auc_cls:.3f})")
    ax.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC — {model_name}\n(macro AUC={r["auc"]:.3f})', fontweight='bold', fontsize=10)
    ax.legend(frameon=False, fontsize=7.5)

plt.suptitle('One-vs-Rest ROC Curves by Model and Class', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('m6_fig03_roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: m6_fig03_roc_curves.png")

---
## 4. Feature Importance Analysis

Feature importance quantifies each variable's contribution to predictive power.
Tree-based models provide intrinsic importance; SHAP provides model-agnostic, game-theoretic attributions.

In [ ]:
# ── Random Forest intrinsic feature importances ───────────────────────────────
rf_clf = test_results['Random Forest']['clf']
fi     = pd.DataFrame({
    'Feature'   : FEATURES,
    'Importance': rf_clf.feature_importances_
}).sort_values('Importance', ascending=True)

# GBT importances for comparison
gb_clf = test_results['Gradient Boosting']['clf']
fi_gb  = pd.DataFrame({
    'Feature'   : FEATURES,
    'Importance': gb_clf.feature_importances_
}).sort_values('Importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, color in zip(
    axes,
    [fi, fi_gb],
    ['Random Forest — Feature Importance (MDI)', 'Gradient Boosting — Feature Importance'],
    ['#2196F3', '#4CAF50']
):
    bars = ax.barh(data['Feature'], data['Importance'], color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, data['Importance']):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8.5)
    ax.set_xlabel('Importance Score')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, data['Importance'].max() + 0.06)

plt.suptitle('Feature Importance Comparison: RF vs Gradient Boosting', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('m6_fig04_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: m6_fig04_feature_importance.png")

print("\nTop 3 features (Random Forest):")
print(fi.sort_values('Importance', ascending=False).head(3).to_string(index=False))

---
## 5. SHAP Explainability — Transparent AI (C2)

SHAP (Lundberg & Lee, 2017) uses cooperative game theory to assign each feature a **contribution score**
for every individual prediction. Unlike standard feature importance, SHAP is:
- **Locally faithful** — explains individual predictions, not just global averages
- **Model-agnostic** — can explain any classifier
- **Theoretically grounded** — the only attribution method satisfying all three Shapley axioms

This aligns with the EU AI Act's requirement for explainable decision systems.

In [ ]:
if SHAP_AVAILABLE:
    # TreeExplainer is optimised for tree-based models — O(TLD) complexity
    explainer   = shap.TreeExplainer(rf_clf)
    shap_values = explainer.shap_values(X_test)   # shape: (n_samples, n_features, n_classes)

    # ── Global SHAP summary — 'Average' class ─────────────────────────────────
    print("Generating SHAP summary plot for 'Average' tip class ...")
    fig, ax = plt.subplots(figsize=(10, 5))
    shap.summary_plot(
        shap_values[:, :, 1],   # class index 1 = 'Average'
        X_test,
        feature_names=FEATURES,
        plot_type='dot',
        show=False,
        max_display=8
    )
    plt.title('SHAP Summary — Class: Average Tipper (10–20%)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('m6_fig05a_shap_summary.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: m6_fig05a_shap_summary.png")

    # ── SHAP bar plot — mean absolute SHAP per class ───────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    class_labels = ['Low', 'Average', 'Generous']
    colors_shap  = ['#FF5722', '#2196F3', '#4CAF50']

    for cls_idx, (cls_name, color) in enumerate(zip(class_labels, colors_shap)):
        mean_shap = np.abs(shap_values[:, :, cls_idx]).mean(axis=0)
        sorted_idx = np.argsort(mean_shap)
        feat_sorted = [FEATURES[i] for i in sorted_idx]
        axes[cls_idx].barh(feat_sorted, mean_shap[sorted_idx], color=color, alpha=0.85, edgecolor='white')
        axes[cls_idx].set_title(f'Mean |SHAP| — {cls_name}', fontweight='bold', fontsize=10)
        axes[cls_idx].set_xlabel('Mean |SHAP value|')

    plt.suptitle('SHAP Mean Absolute Values by Tip-Generosity Class', fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('m6_fig05b_shap_by_class.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: m6_fig05b_shap_by_class.png")

    # ── Waterfall plot for one example prediction ──────────────────────────────
    print("\nWaterfall plot — single prediction explanation:")
    shap_exp = shap.Explanation(
        values       = shap_values[0, :, 2],   # first test record, Generous class
        base_values  = explainer.expected_value[2],
        data         = X_test[0],
        feature_names= FEATURES
    )
    shap.waterfall_plot(shap_exp, max_display=8, show=False)
    plt.title('SHAP Waterfall — Prediction #0 (Generous class)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('m6_fig05c_shap_waterfall.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: m6_fig05c_shap_waterfall.png")

else:
    # ── Fallback: permutation importance ─────────────────────────────────────
    print("SHAP not available — computing Permutation Importance as fallback ...")
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(rf_clf, X_test, y_test, n_repeats=30, random_state=SEED)
    perm_df = pd.DataFrame({'Feature': FEATURES,
                             'Mean': perm.importances_mean,
                             'Std' : perm.importances_std}).sort_values('Mean')
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(perm_df['Feature'], perm_df['Mean'],
            xerr=perm_df['Std'], color='#2196F3', alpha=0.85, edgecolor='white')
    ax.set_title('Permutation Importance (SHAP fallback)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('m6_fig05_permutation_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: m6_fig05_permutation_importance.png")

---
## 6. K-Means Customer Segmentation (C3)

Clustering moves beyond prediction to **discovery** — finding natural groupings in the data
that were not defined a priori. Each cluster represents a distinct **customer archetype**
with its own tipping and spending fingerprint.

In [ ]:
# ── Elbow method to find optimal K ───────────────────────────────────────────
# Cluster on: total_bill, tip_pct, size, bill_per_person, is_weekend
CLUSTER_FEATURES = ['total_bill', 'tip_pct', 'size', 'bill_per_person', 'is_weekend']
X_clust = scaler.fit_transform(df[CLUSTER_FEATURES].values)

inertias = []
sil_scores = []
K_range = range(2, 9)

from sklearn.metrics import silhouette_score
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=15)
    km.fit(X_clust)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), inertias, 'o-', color='#2196F3', lw=2, markersize=7)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xticks(list(K_range))

axes[1].plot(list(K_range), sil_scores, 's-', color='#4CAF50', lw=2, markersize=7)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score by K', fontweight='bold')
axes[1].set_xticks(list(K_range))

best_k = list(K_range)[np.argmax(sil_scores)]
axes[1].axvline(best_k, color='#FF5722', ls='--', lw=1.5, label=f'Best K={best_k}')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig('m6_fig06_elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Optimal K = {best_k} (Silhouette = {max(sil_scores):.4f})")
print("Saved: m6_fig06_elbow_silhouette.png")

In [ ]:
# ── Fit final K-Means model ───────────────────────────────────────────────────
OPTIMAL_K = best_k
km_final  = KMeans(n_clusters=OPTIMAL_K, random_state=SEED, n_init=15)
df['cluster'] = km_final.fit_predict(X_clust)

# ── Cluster profile table ─────────────────────────────────────────────────────
cluster_profile = df.groupby('cluster').agg(
    n              = ('tip', 'count'),
    avg_bill       = ('total_bill', 'mean'),
    avg_tip        = ('tip', 'mean'),
    avg_tip_pct    = ('tip_pct', 'mean'),
    avg_size       = ('size', 'mean'),
    pct_weekend    = ('is_weekend', 'mean'),
    pct_dinner     = ('time_encoded', 'mean'),
    pct_smoker     = ('smoker_encoded', 'mean'),
    pct_male       = ('sex_encoded', 'mean'),
).round(3)

# Assign archetype names based on profile
archetype_names = {}
for c in range(OPTIMAL_K):
    row = cluster_profile.loc[c]
    if row['avg_tip_pct'] >= 20:
        archetype_names[c] = f"Cluster {c}: Generous Weekend Diners"
    elif row['avg_bill'] >= cluster_profile['avg_bill'].mean() + 3:
        archetype_names[c] = f"Cluster {c}: High-Spend Groups"
    elif row['avg_tip_pct'] < 14:
        archetype_names[c] = f"Cluster {c}: Conservative Tippers"
    else:
        archetype_names[c] = f"Cluster {c}: Mid-Range Regulars"

cluster_profile['Archetype'] = [archetype_names[c] for c in cluster_profile.index]
print("Cluster Profile Table:")
print(cluster_profile.to_string())

In [ ]:
# ── PCA visualisation of clusters ─────────────────────────────────────────────
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_clust)
var_explained = pca.explained_variance_ratio_

cluster_colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800'][:OPTIMAL_K]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter by cluster
for c in range(OPTIMAL_K):
    mask = df['cluster'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=cluster_colors[c], s=55, alpha=0.75, edgecolors='grey', lw=0.3,
                    label=archetype_names.get(c, f'Cluster {c}'))

# Plot centroids
centroids_pca = pca.transform(km_final.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                marker='X', s=180, color='black', zorder=10, label='Centroids')

axes[0].set_xlabel(f'PC1 ({var_explained[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({var_explained[1]*100:.1f}% variance)')
axes[0].set_title('K-Means Clusters — PCA Projection', fontweight='bold')
axes[0].legend(frameon=False, fontsize=8)

# Scatter coloured by tip generosity (for comparison)
gen_colors = {'Low': '#FF5722', 'Average': '#FFC107', 'Generous': '#4CAF50'}
for cat, color in gen_colors.items():
    mask = df['tip_category'] == cat
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    color=color, s=55, alpha=0.75, edgecolors='grey', lw=0.3, label=cat)

axes[1].set_xlabel(f'PC1 ({var_explained[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({var_explained[1]*100:.1f}% variance)')
axes[1].set_title('Same Space — Coloured by Tip Category', fontweight='bold')
axes[1].legend(frameon=False, fontsize=9, title='Tip Category')

plt.tight_layout()
plt.savefig('m6_fig07_pca_clusters.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved: m6_fig07_pca_clusters.png")
print(f"PCA variance explained: PC1={var_explained[0]*100:.1f}%, PC2={var_explained[1]*100:.1f}%")

In [ ]:
# ── Radar/Spider chart — cluster archetype fingerprints ───────────────────────
radar_features = ['avg_bill', 'avg_tip_pct', 'avg_size', 'pct_weekend', 'pct_dinner', 'pct_smoker']
radar_labels   = ['Avg Bill ($)', 'Avg Tip %', 'Avg Size', '% Weekend', '% Dinner', '% Smoker']

# Normalise radar data 0–1
radar_data = cluster_profile[radar_features].values.astype(float)
mins = radar_data.min(axis=0)
maxs = radar_data.max(axis=0)
radar_norm = (radar_data - mins) / (maxs - mins + 1e-9)

N      = len(radar_labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, axes = plt.subplots(1, OPTIMAL_K, subplot_kw=dict(polar=True),
                          figsize=(4 * OPTIMAL_K, 4))
if OPTIMAL_K == 1:
    axes = [axes]

for i, (ax, color) in enumerate(zip(axes, cluster_colors)):
    vals = radar_norm[i].tolist() + radar_norm[i][:1].tolist()
    ax.plot(angles, vals, color=color, lw=2)
    ax.fill(angles, vals, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=8)
    ax.set_yticklabels([])
    ax.set_title(archetype_names.get(i, f'Cluster {i}').replace(f'Cluster {i}: ', ''),
                 fontweight='bold', fontsize=9, pad=15)

plt.suptitle('Customer Archetype Radar Charts', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('m6_fig08_radar_archetypes.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: m6_fig08_radar_archetypes.png")

---
## 7. Gradient Boosting Regression — Continuous Tip Prediction

Extending from classification to regression: predict the actual tip dollar amount,
enabling the system to estimate expected revenue for any given table configuration.

In [ ]:
# ── Regression target: tip ($) ────────────────────────────────────────────────
y_reg = df['tip'].values

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_scaled, y_reg, test_size=0.25, random_state=SEED)

gb_reg = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.07, max_depth=4,
    subsample=0.8, random_state=SEED)

gb_reg.fit(X_tr_r, y_tr_r)
y_pred_reg = gb_reg.predict(X_te_r)

rmse = np.sqrt(mean_squared_error(y_te_r, y_pred_reg))
mae  = mean_absolute_error(y_te_r, y_pred_reg)
r2   = r2_score(y_te_r, y_pred_reg)

print("Gradient Boosting Regression — Tip Amount Prediction")
print("=" * 50)
print(f"  RMSE : ${rmse:.4f}")
print(f"  MAE  : ${mae:.4f}")
print(f"  R²   : {r2:.4f}")
print(f"  Baseline RMSE (predict mean): "
      f"${np.sqrt(mean_squared_error(y_te_r, np.full_like(y_te_r, y_tr_r.mean()))):.4f}")
print("=" * 50)

In [ ]:
# ── Regression diagnostic plots ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
axes[0].scatter(y_te_r, y_pred_reg, color='#2196F3', alpha=0.65, s=50, edgecolors='grey', lw=0.3)
lims = [min(y_te_r.min(), y_pred_reg.min()) - 0.3,
        max(y_te_r.max(), y_pred_reg.max()) + 0.3]
axes[0].plot(lims, lims, 'k--', lw=1.2, label='Perfect prediction')
axes[0].set_xlabel('Actual Tip ($)')
axes[0].set_ylabel('Predicted Tip ($)')
axes[0].set_title(f'Actual vs Predicted Tip\n(R²={r2:.3f}, RMSE=${rmse:.3f})', fontweight='bold')
axes[0].legend(frameon=False)

# Residuals
residuals = y_te_r - y_pred_reg
axes[1].scatter(y_pred_reg, residuals, color='#FF5722', alpha=0.65, s=50, edgecolors='grey', lw=0.3)
axes[1].axhline(0, color='#333', lw=1.2, ls='--')
axes[1].axhline(residuals.std(), color='#4CAF50', lw=1, ls=':', label=f'+1 SD (${residuals.std():.2f})')
axes[1].axhline(-residuals.std(), color='#4CAF50', lw=1, ls=':', label=f'-1 SD')
axes[1].set_xlabel('Predicted Tip ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot — Random Scatter is Ideal', fontweight='bold')
axes[1].legend(frameon=False)

plt.tight_layout()
plt.savefig('m6_fig09_regression_diagnostics.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: m6_fig09_regression_diagnostics.png")

---
## 8. Learning Curves — Diagnosing Bias vs Variance

In [ ]:
# ── Learning curves for RF and GBT ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, clf), color in zip(
    axes,
    [('Random Forest', models['Random Forest']),
     ('Gradient Boosting', models['Gradient Boosting'])],
    ['#2196F3', '#4CAF50']
):
    train_sizes, train_scores, val_scores = learning_curve(
        clf, X_scaled, y, cv=5, scoring='accuracy',
        train_sizes=np.linspace(0.15, 1.0, 10),
        random_state=SEED, n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    val_mean   = val_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_std    = val_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'o-', color=color, lw=2, label='Training score')
    ax.plot(train_sizes, val_mean,   's--', color=color, lw=2, alpha=0.6, label='CV score')
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                    alpha=0.1, color=color)
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                    alpha=0.1, color=color)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Learning Curve — {name}', fontweight='bold')
    ax.set_ylim(0.3, 1.05)
    ax.legend(frameon=False)

plt.tight_layout()
plt.savefig('m6_fig10_learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: m6_fig10_learning_curves.png")

---
## 9. Integrated System Summary

All milestones are now combined into a single reproducible pipeline.

In [ ]:
# ── Final integrated summary ───────────────────────────────────────────────────
print("\n" + "="*70)
print("  INTEGRATED SYSTEM SUMMARY — Sprint 2, Milestones 1–6")
print("="*70)
print(f"\n  Dataset: Restaurant Tips (seaborn) | {df.shape[0]} records, {df.shape[1]} features")
print(f"  Features engineered : 8 (from 7 original)")
print(f"  Hypotheses tested   : 5 (t-tests, ANOVA, chi-square)")
print(f"  Visualisation types : 10+ across Milestones 3 & 6")
print(f"  ML models trained   : 3 classifiers + 1 regressor")
print(f"  Customer clusters   : {OPTIMAL_K}")
print()
print("  Model Leaderboard:")
print(f"  {'Model':<25} {'CV Acc':>8} {'Test Acc':>10} {'ROC-AUC':>10}")
print("  " + "-"*55)
for name in models:
    cv_m   = cv_results[name].mean()
    te_acc = test_results[name]['acc']
    te_auc = test_results[name]['auc']
    auc_str = f"{te_auc:.4f}" if te_auc else "N/A"
    best_flag = " ◀ BEST" if name == best_model_name else ""
    print(f"  {name:<25} {cv_m:.4f}   {te_acc:.4f}     {auc_str}{best_flag}")

print()
print("  Regression (Tip Amount):")
print(f"  Gradient Boosting Regressor | RMSE=${rmse:.3f} | MAE=${mae:.3f} | R²={r2:.3f}")
print()
print("  Key Insight: 'total_bill' is the dominant predictor of tip generosity.")
print("  Customer clusters reveal actionable archetypes for marketing & operations.")
print("  SHAP explainability bridges the gap between prediction and decision support.")
print("="*70)

---
## 10. Research Insights & Conclusions

### 10.1 Predictive Modelling Findings

| Finding | Evidence |
|---|---|
| **Total bill is the single strongest predictor** of tip category | Highest feature importance in RF and GBT; SHAP confirms positive direction |
| **Weekend and dinner flags** add moderate signal | Feature importance rank 3–4 in both tree models |
| **Demographic features** (sex, smoker) have near-zero predictive power | Consistent with Milestone 4 chi-square non-significant results |
| **GBT outperforms RF on generalisation** | Lower train–test gap in learning curves |

### 10.2 Clustering Findings

| Cluster Archetype | Characteristics | Business Action |
|---|---|---|
| Generous Weekend Diners | High tip %, weekend, dinner | Loyalty programme priority |
| High-Spend Groups | Large bills, big parties | Service charge evaluation |
| Conservative Tippers | Low tip %, small bills | Upsell training focus |
| Mid-Range Regulars | Average across all metrics | Retention via consistency |

### 10.3 Limitations

- **Small dataset** (n=244): limits statistical power and generalisability; model confidence intervals are wide
- **Cross-sectional data**: no time dimension to track customer behaviour change
- **Feature ceiling**: the dataset lacks contextual variables (server, cuisine type, occasion)
- **Ecological validity**: findings are specific to one restaurant context

### 10.4 Future Work

1. **Streaming analytics**: extend pipeline to real-time POS data using Apache Kafka + Flink
2. **Large Language Model integration**: NLP on customer reviews to enrich tip prediction features
3. **Counterfactual explanations**: complement SHAP with DiCE for actionable what-if guidance
4. **Multi-restaurant federation**: train on pooled data with federated learning to preserve privacy
5. **Reinforcement learning**: optimise table assignment and upsell timing using tip signals as reward